In [ ]:
# record_auto_stop関数 (record関数の自動停止版、無音を検知するまで録音)
# Chapter 3に追加したものとほぼ同じ（最後の戻り値だけ変更）

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from pydub import AudioSegment
import soundfile as sf
import io

error_count = 0

def record_auto_stop(
  SILENCE_RMS,      # 無音レベルの大きさの指標
  SILENCE_SEC,      # 秒. 無音がこの時間続いたら録音終了
):
  display(Javascript('''

    // メッセージ表示
    const message = (text) => {
      const domId = 'message';
      const output = document.querySelector('#output-area');
      let target = document.querySelector(`#${domId}`);
      if (!target) {
        target = document.createElement('div');
        target.id = domId;
        output.insertBefore(target, output.firstChild);
      }
      target.innerHTML += `${text}<br>`;
    };

    // 音量の指標を計算
    const calculateRMS = (data) => {
      let sum = 0;
      for (let i = 0; i < data.length; i++) {
        const normalized = data[i] / 128 - 1;
        sum += normalized * normalized;
      }
      return Math.sqrt(sum / data.length);
    };

    async function recordAndAutoStop(SILENCE_RMS, SILENCE_SEC) {
      // マイク使用可否チェック
      let stream = null;
      try {
        stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      } catch(e) {
        return;
      }

      const audioContext = new AudioContext();
      const source = audioContext.createMediaStreamSource(stream);
      const analyser = audioContext.createAnalyser();
      source.connect(analyser);

      analyser.fftSize = 2048;
      const bufferLength = analyser.fftSize;
      const dataArray = new Uint8Array(bufferLength);
      let silenceStart = performance.now();

      const chunks = [];
      const recorder = new MediaRecorder(stream);
      recorder.ondataavailable = e => {
        message('録音中');
        chunks.push(e.data);
      };

      // 無音検知
      const detectSilence = () => {
        analyser.getByteTimeDomainData(dataArray);
        if (calculateRMS(dataArray) < SILENCE_RMS) {
          const now = performance.now();
          if (now - silenceStart > (SILENCE_SEC * 10**3)) {
            recorder.stop();
            return;
          }
        } else {
          silenceStart = performance.now();
          if (recorder.state === 'inactive') {
            recorder.start();
          }
        }
        requestAnimationFrame(detectSilence);
      }

      const fr = new FileReader();
      message('録音開始');
      recorder.onstop = e => {
        message('録音終了');
        fr.readAsDataURL(new Blob(chunks))
      };
      recorder.start(1000);
      detectSilence();

      return await new Promise(resolve => {
        fr.onloadend = () => resolve(fr.result);
      });
    };
  '''))

  data = eval_js(f'recordAndAutoStop({SILENCE_RMS}, {SILENCE_SEC})')

  # 簡易エラー処理
  global error_count
  if data == None:
    if error_count == 0:
      error_count += 1
      print('''
        ブラウザ画面の中央にマイク使用を求める表示があると思います。
        許可を選択し、もう一度セルを実行して下さい。
        また別のマイク使用を求めるポップアップが出たら、許可して下さい。
      '''.replace(' ', ''))
    else:
      print('マイクを使用できません')
    return [], 0

  # WAV形式に統一
  buffer = io.BytesIO()
  AudioSegment.from_file(
    io.BytesIO(b64decode(data.split(',')[1]))
  ).export(buffer, format="wav")
  buffer.seek(0)
  return buffer

print('録音準備完了')


In [ ]:
# Chapter 4 > 4-2-2 (p.112) Colab版
# 独自のrecord_auto_stop関数を利用

SILENCE_RMS = 0.01 # 無音レベルの指標（環境音が大きければ増やす）
SILENCE_SEC = 2    # 秒. 無音がこの時間続いたら録音終了

!pip install SpeechRecognition # 2回目以降の実行ではコメントアウトしてもよい
import speech_recognition as sr
import soundfile as sf

memory_file = record_auto_stop(SILENCE_RMS, SILENCE_SEC)

try:
  r = sr.Recognizer()
  with sr.AudioFile(memory_file) as source:
    audio = r.record(source)
  recognized_text = r.recognize_google(audio, language='ja')
  print(f'音声認識結果「{recognized_text}」')
except sr.UnknownValueError:
  print('認識できません')


In [ ]:
# マイク音声取得とテキスト認識をまとめて行ったセルを関数化

!pip install SpeechRecognition # 2回目以降の実行ではコメントアウトしてもよい
import speech_recognition as sr
import soundfile as sf

def mic_to_recognize(SILENCE_RMS, SILENCE_SEC): # 無音判定する音量, 秒数
  """マイク音声取得とテキスト認識をまとめて行う関数"""

  # 必要な関数を使えるかチェック
  try:
    record_auto_stop
  except NameError:
    print('先にrecord_auto_stop関数のセルの実行が必要です')
    return ''

  memory_file = record_auto_stop(SILENCE_RMS, SILENCE_SEC)

  try:
    r = sr.Recognizer()
    with sr.AudioFile(memory_file) as source:
      audio = r.record(source)
    return r.recognize_google(audio, language='ja')
  except sr.UnknownValueError:
    print('認識できません')
    return ''


In [ ]:
# mic_to_ecognize関数のテスト

text = mic_to_recognize(0.01, 2) # 無音判定する音量, 秒数
if text:
  print(f'音声認識結果「{text}」')


In [ ]:
# 本 4-3 (p.114-120) で作る関数の完成形

def tamego_to_teineigo(text):
  """タメ口を丁寧語に変換する関数"""

  # 変換パターン

  patterns = {
    'だね': 'ですね',
    'こんにちは': 'ごきげんよう',
    'だ': 'です'
  }

  # テキストをスペースで分離する
  sentences = text.split(' ')

  # 変換
  teineigo_sentences = []
  for sentence in sentences:
    for pattern, replacement in patterns.items():
      sentence = sentence.replace(pattern, replacement)
    teineigo_sentences.append(sentence)

  joined_text = ' '.join(teineigo_sentences)

  return joined_text


In [ ]:
# 本にない独自の工夫
# 2つの関数を利用して「マイク音声取得→テキスト認識→丁寧語変換」を短いコードで行う
# 学習会が扱うChapter 4はいったんここまでとしChapter 5を先にしたい

text = mic_to_recognize(0.01, 2) # 無音判定する音量, 秒数
if text:
  print(f'音声認識結果「{text}」')
  print(f'丁寧語変換結果「{tamego_to_teineigo(text)}」')
